# 07. Data Cleaning with Pandas

## Objective
Identify and handle missing values, duplicate records, incorrect data types, inconsistent values, and potential outliers using the Titanic dataset.

**Suggested location:** `EDA/07_Data_Cleaning.ipynb`

Run cells from top to bottom. The original data is preserved in `titanic_raw`; cleaning is performed on a copy called `titanic_clean`.

## 1. Import Libraries and Load the Dataset

We load a local CSV when available; otherwise, the notebook reads the public Titanic CSV online. Internet access is needed for the fallback.

In [1]:
from pathlib import Path
import numpy as np
import pandas as pd

data_path = Path("data/titanic.csv")
if data_path.exists():
    titanic_raw = pd.read_csv(data_path)
    print(f"Loaded local dataset: {data_path}")
else:
    url = "https://raw.githubusercontent.com/mwaskom/seaborn-data/master/titanic.csv"
    titanic_raw = pd.read_csv(url)
    print("Loaded Titanic dataset from the public Seaborn example-data repository.")

print("Original shape:", titanic_raw.shape)
titanic_raw.head()

Loaded Titanic dataset from the public Seaborn example-data repository.
Original shape: (891, 15)


,survived,pclass,sex,age,sibsp,parch,fare,embarked,class,who,adult_male,deck,embark_town,alive,alone
0,0,3,male,22.0,1,0,7.2500,S,Third,man,True,NaN,Southampton,no,False
1,1,1,female,38.0,1,0,71.2833,C,First,woman,False,C,Cherbourg,yes,False
2,1,3,female,26.0,0,0,7.9250,S,Third,woman,False,NaN,Southampton,yes,True
3,1,1,female,35.0,1,0,53.1000,S,First,woman,False,C,Southampton,yes,False
4,0,3,male,35.0,0,0,8.0500,S,Third,man,True,NaN,Southampton,no,True


## 2. Create a Working Copy

Keeping the original unchanged lets us compare the dataset before and after cleaning.

In [3]:
titanic_clean = titanic_raw.copy()
print("Raw shape:", titanic_raw.shape)
print("Working-copy shape:", titanic_clean.shape)

Raw shape: (891, 15)
Working-copy shape: (891, 15)


## 3. Identify Missing Values

Missing values can affect calculations. We count missing values and calculate their percentage in each column.

In [4]:
missing_summary = pd.DataFrame({
    "Missing Count": titanic_clean.isna().sum(),
    "Missing Percentage": (titanic_clean.isna().mean() * 100).round(2)
})
missing_summary.sort_values("Missing Count", ascending=False)

,Missing Count,Missing Percentage
deck,688,77.22
age,177,19.87
embarked,2,0.22
embark_town,2,0.22
sex,0,0.00
pclass,0,0.00
survived,0,0.00
fare,0,0.00
parch,0,0.00
sibsp,0,0.00


### Handle selected missing values

- `age`: fill with the median, a value less sensitive to extremes than the mean.
- `embarked`: fill with the most frequent category (mode).
- `deck`: retain missing values because the information is substantially incomplete; don't invent values.
- Other columns: leave missing values unless a specific analysis calls for a documented treatment.

In [5]:
age_median = titanic_clean["age"].median()
titanic_clean["age"] = titanic_clean["age"].fillna(age_median)

embarked_mode = titanic_clean["embarked"].mode()
if not embarked_mode.empty:
    titanic_clean["embarked"] = titanic_clean["embarked"].fillna(embarked_mode.iloc[0])

print("Median age used:", age_median)
print("Embarked mode:", embarked_mode.iloc[0] if not embarked_mode.empty else "No mode")
display(titanic_clean.isna().sum().to_frame("Missing Count"))

Median age used: 28.0
Embarked mode: S


,Missing Count
survived,0
pclass,0
sex,0
age,0
sibsp,0
parch,0
fare,0
embarked,0
class,0
who,0


**Why retain missing `deck` values?** The column has extensive missingness. Dropping every row with a missing deck could discard many records, while filling it would create information that was never collected.

## 4. Identify and Handle Duplicate Records

We inspect exact duplicate rows. Similar passenger details do not necessarily mean the records are duplicates, so we only remove rows that are identical across every column.

In [6]:
duplicate_count = titanic_clean.duplicated().sum()
print("Exact duplicate rows:", duplicate_count)

titanic_clean = titanic_clean.drop_duplicates().copy()
print("Shape after removing exact duplicates:", titanic_clean.shape)
print("Remaining exact duplicates:", titanic_clean.duplicated().sum())

Exact duplicate rows: 110
Shape after removing exact duplicates: (781, 15)
Remaining exact duplicates: 0


## 5. Check and Correct Data Types

Inspect the types first. Selected text-based variables are converted to Pandas `category` dtype. The `survived` column remains numeric (0 = did not survive, 1 = survived).

In [7]:
print("Before conversion:")
print(titanic_clean.dtypes)

categorical_columns = ["sex", "embarked", "class", "who", "deck", "embark_town", "alive"]
for column in categorical_columns:
    if column in titanic_clean.columns:
        titanic_clean[column] = titanic_clean[column].astype("category")

print("\nAfter conversion:")
print(titanic_clean.dtypes)

Before conversion:
survived         int64
pclass           int64
sex             object
age            float64
sibsp            int64
parch            int64
fare           float64
embarked        object
class           object
who             object
adult_male        bool
deck            object
embark_town     object
alive           object
alone             bool
dtype: object

After conversion:
survived          int64
pclass            int64
sex            category
age             float64
sibsp             int64
parch             int64
fare            float64
embarked       category
class          category
who            category
adult_male         bool
deck           category
embark_town    category
alive          category
alone              bool
dtype: object


## 6. Standardize Inconsistent Values

Inspect category labels before changing them. We trim whitespace and use title case for selected text columns, then restore the categorical dtype. The source data is already fairly standardized, so this is a conservative demonstration.

In [8]:
text_columns = ["sex", "embarked", "class", "who", "embark_town", "alive"]

print("Values before standardization:")
for column in text_columns:
    if column in titanic_clean.columns:
        print(f"{column}: {titanic_clean[column].dropna().unique().tolist()}")

for column in text_columns:
    if column in titanic_clean.columns:
        cleaned = titanic_clean[column].astype("string").str.strip().str.title()
        titanic_clean[column] = cleaned.astype("category")

print("\nValues after standardization:")
for column in text_columns:
    if column in titanic_clean.columns:
        print(f"{column}: {titanic_clean[column].dropna().unique().tolist()}")

Values before standardization:
sex: ['male', 'female']
embarked: ['S', 'C', 'Q']
class: ['Third', 'First', 'Second']
who: ['man', 'woman', 'child']
embark_town: ['Southampton', 'Cherbourg', 'Queenstown']
alive: ['no', 'yes']

Values after standardization:
sex: ['Male', 'Female']
embarked: ['S', 'C', 'Q']
class: ['Third', 'First', 'Second']
who: ['Man', 'Woman', 'Child']
embark_town: ['Southampton', 'Cherbourg', 'Queenstown']
alive: ['No', 'Yes']


## 7. Detect Potential Outliers Using IQR

The Interquartile Range (IQR) method flags values below Q1 − 1.5 × IQR or above Q3 + 1.5 × IQR.

**A flag does not prove an error.** High fares or ages may be genuine, so we identify potential outliers but do not automatically delete them.

In [9]:
def iqr_outlier_summary(data, column):
    values = data[column].dropna()
    q1 = values.quantile(0.25)
    q3 = values.quantile(0.75)
    iqr = q3 - q1
    lower = q1 - 1.5 * iqr
    upper = q3 + 1.5 * iqr
    outliers = values[(values < lower) | (values > upper)]
    return {
        "Column": column, "Q1": q1, "Q3": q3, "IQR": iqr,
        "Lower Bound": lower, "Upper Bound": upper,
        "Potential Outliers": len(outliers)
    }

outlier_summary = pd.DataFrame([
    iqr_outlier_summary(titanic_clean, "age"),
    iqr_outlier_summary(titanic_clean, "fare")
])
outlier_summary

,Column,Q1,Q3,IQR,Lower Bound,Upper Bound,Potential Outliers
0,age,22.00,36.0000,14.0000,1.0000,57.000,39
1,fare,8.05,34.0208,25.9708,-30.9062,72.977,102


### Outlier treatment decision

We retain flagged age and fare values because they may represent real passengers and ticket prices. Removing them without contextual evidence could distort the analysis. Robust statistics or a justified filter can be considered later if a research question requires it.

## 8. Compare Before and After Cleaning

In [10]:
comparison = pd.DataFrame({
    "Raw": {
        "Rows": titanic_raw.shape[0],
        "Columns": titanic_raw.shape[1],
        "Missing Cells": int(titanic_raw.isna().sum().sum()),
        "Exact Duplicate Rows": int(titanic_raw.duplicated().sum())
    },
    "Cleaned": {
        "Rows": titanic_clean.shape[0],
        "Columns": titanic_clean.shape[1],
        "Missing Cells": int(titanic_clean.isna().sum().sum()),
        "Exact Duplicate Rows": int(titanic_clean.duplicated().sum())
    }
})
comparison

,Raw,Cleaned
Rows,891,781
Columns,15,15
Missing Cells,869,581
Exact Duplicate Rows,107,0


### Remaining missing values

In [11]:
remaining_missing = titanic_clean.isna().sum()
remaining_missing[remaining_missing > 0].sort_values(ascending=False)

,0
deck,579
embark_town,2


## 9. Save the Cleaned Dataset

Save to a separate CSV so the raw dataset remains unchanged.

In [12]:
output_dir = Path("data")
output_dir.mkdir(parents=True, exist_ok=True)
cleaned_path = output_dir / "titanic_cleaned.csv"
titanic_clean.to_csv(cleaned_path, index=False)
print(f"Cleaned dataset saved to: {cleaned_path.resolve()}")

Cleaned dataset saved to: /content/data/titanic_cleaned.csv


## 10. Summary of Cleaning Decisions

| Issue | Action | Reason |
|---|---|---|
| Missing age | Filled with median | Robust summary; retains rows |
| Missing embarkation port | Filled with mode | Most frequent observed category |
| Missing deck | Retained | Too incomplete to infer reliably |
| Exact duplicate rows | Removed | Prevents identical repeated records |
| Data types | Selected text columns converted to category | Makes categorical variables explicit |
| Inconsistent text | Trimmed and title-cased | Improves label consistency |
| Potential outliers | Flagged, not removed | Unusual values may be valid |

## Conclusion

We preserved the original Titanic dataset, cleaned a working copy, documented missing-value and duplicate handling, standardized selected data types and text labels, flagged potential outliers, and saved the cleaned dataset for the next EDA practical.